# Experiment 2: Counterfactual Sensitivity

Family-level analysis: fragility rate, severity distribution, pattern analysis.

**Primary analysis unit**: family (not instance). Family members are correlated, not IID.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
from utils import load_all_results

from bcbench.analysis.aggregator import build_families
from bcbench.analysis.family import FamilyOutcome
from bcbench.analysis.metrics import (
    consistency_distribution,
    correctness_drop,
    family_type_distribution,
    fragility_rate,
    mean_severity,
)
from bcbench.analysis.sensitivity import (
    build_patch_pairs,
    mean_sensitivity,
    median_sensitivity,
    patch_distance,
    reaction_distribution,
    sensitivity_score,
)
from bcbench.results.base import ExecutionBasedEvaluationResult

# All four CF setups now carry base (5 runs) + CF results, so family-level metrics
# (fragility, four-way consistency, correctness drop) are comparable across models.
SETUPS = {
    "copilot-cf-opus-4-6": "Claude Opus 4.6",
    "copilot-cf-sonnet-4-6": "Claude Sonnet 4.6",
    "copilot-cf-gpt-5-4": "GPT-5-4",
    "copilot-cf-gpt-5-3-codex": "GPT-5-3 Codex",
}
# Sensitivity / patch-distance need the base generated patch; only Opus retains
# per-instance base output (the other models' base artifacts expired, leaving
# resolved-only results recovered from the leaderboard), so structural metrics
# are computed for Opus only.
SENSITIVITY_SETUP = "copilot-cf-opus-4-6"

all_results = load_all_results(category="bug-fix")


def df_to_results(df: pd.DataFrame) -> list[ExecutionBasedEvaluationResult]:
    results = []
    for _, row in df.iterrows():
        results.append(
            ExecutionBasedEvaluationResult(
                instance_id=row["instance_id"],
                project=row.get("project", ""),
                model=row.get("model", "unknown"),
                agent_name="GitHub Copilot",
                category="cf",
                resolved=bool(row.get("resolved", False)),
                build=bool(row.get("build", False)),
                output=row.get("output", ""),
            )
        )
    return results


families_by_model: dict[str, list[FamilyOutcome]] = {}
results_by_model: dict[str, list[ExecutionBasedEvaluationResult]] = {}
for setup, label in SETUPS.items():
    if setup not in all_results:
        print(f"WARNING: {setup} not found")
        continue
    res = df_to_results(all_results[setup])
    results_by_model[label] = res
    families_by_model[label] = build_families(res)
    n_base = sum(1 for r in res if "__cf-" not in r.instance_id)
    n_cf = sum(1 for r in res if "__cf-" in r.instance_id)
    print(f"{label}: {len(res)} results (base={n_base}, cf={n_cf}) -> {len(families_by_model[label])} families")


# Gold test patches + model-generated patches for structural (sensitivity) metrics
def _load_gold_test_patches() -> dict[str, str]:
    root = Path.cwd()
    while root != root.parent and not (root / "dataset" / "counterfactual.jsonl").exists():
        root = root.parent
    gold: dict[str, str] = {}
    for name in ("bcbench.jsonl", "counterfactual.jsonl"):
        p = root / "dataset" / name
        if not p.exists():
            continue
        for line in p.read_text(encoding="utf-8").splitlines():
            if line.strip():
                o = json.loads(line)
                if o.get("test_patch") is not None:
                    gold[o["instance_id"]] = o["test_patch"]
    return gold


gold_test_patches = _load_gold_test_patches()
# Sensitivity needs base generated patches. Build pairs only for models whose base
# outputs are present (Opus always; Sonnet & GPT-5.3-Codex after the base re-run).
# Models with resolved-only recovered base (e.g. GPT-5-4) are skipped automatically.
pairs_by_model: dict[str, list] = {}
for label, res in results_by_model.items():
    gen = {r.instance_id: (r.output or "") for r in res}
    passed = {r.instance_id: r.resolved for r in res}
    base_has_output = any(("__cf-" not in iid) and gen.get(iid) for iid in gen)
    if not base_has_output:
        continue
    p = build_patch_pairs(gen, gold_test_patches, passed)
    if p:
        pairs_by_model[label] = p

print(f"gold test patches: {len(gold_test_patches)}")
print("sensitivity models:", {k: len(v) for k, v in pairs_by_model.items()})

Claude Opus 4.6: 626 results (base=505, cf=121) -> 71 families
Claude Sonnet 4.6: 222 results (base=101, cf=121) -> 71 families
GPT-5-4: 626 results (base=505, cf=121) -> 71 families
GPT-5-3 Codex: 222 results (base=101, cf=121) -> 71 families
gold test patches: 222
sensitivity models: {'Claude Opus 4.6': 121, 'Claude Sonnet 4.6': 121, 'GPT-5-3 Codex': 121}


## Family Outcome Table

Each row = one family. Columns: family_id, layer, base, cf1..cfN, pattern, type, fragile, severity.

In [2]:
def families_to_df(families: list[FamilyOutcome]) -> pd.DataFrame:
    rows = []
    for f in families:
        row = {
            "family_id": f.family_id,
            "layer": f.failure_layer.value if f.failure_layer else "unclassified",
            "base": int(f.base.passed),
            "pattern": str(f.pattern),
            "type": f.family_type.value,
            "fragile": int(f.is_fragile),
            "severity": f.severity,
            "cf_total": f.cf_total,
            "cf_fail_count": f.cf_fail_count,
        }
        for i, cf in enumerate(f.cfs):
            row[f"cf{i + 1}"] = int(cf.passed)
        rows.append(row)
    return pd.DataFrame(rows)


print("Family outcome table builder ready. Load results to populate.")

Family outcome table builder ready. Load results to populate.


## Family Type Distribution

In [3]:
def plot_family_type_distribution(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        dist = family_type_distribution(families)
        total = sum(dist.values())
        for ftype, count in dist.items():
            rows.append(
                {
                    "Model": model,
                    "Family Type": ftype,
                    "Count": count,
                    "Proportion (%)": round(count / total * 100, 1) if total else 0,
                }
            )

    df = pd.DataFrame(rows)
    fig = px.bar(
        df,
        x="Model",
        y="Proportion (%)",
        color="Family Type",
        title="Family Type Distribution by Model",
        barmode="stack",
        text_auto=True,
        color_discrete_map={
            "stable-correct": "#2ecc71",
            "fragile": "#e74c3c",
            "unsolved": "#95a5a6",
            "inconsistent": "#f39c12",
        },
    )
    fig.show()
    return df


print("plot_family_type_distribution() ready")

plot_family_type_distribution() ready


## Fragility Rate (Main Thesis Figure)

In [4]:
def plot_fragility_rate(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        rate = fragility_rate(families)
        sev = mean_severity(families)
        rows.append(
            {
                "Model": model,
                "Fragility Rate (%)": round(rate * 100, 1),
                "Mean Severity": round(sev, 3) if sev is not None else None,
            }
        )

    df = pd.DataFrame(rows)
    fig = px.bar(
        df,
        x="Model",
        y="Fragility Rate (%)",
        title="Family Fragility Rate: P(CF fail | Base correct)",
        text_auto=True,
        color_discrete_sequence=["#e74c3c"],
    )
    fig.update_layout(yaxis_range=[0, 100])
    fig.show()
    return df


print("plot_fragility_rate() ready")

plot_fragility_rate() ready


## Severity Distribution

In [5]:
def plot_severity_distribution(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        for f in families:
            if f.severity is not None:
                rows.append({"Model": model, "Severity": f.severity, "Type": f.family_type.value})

    df = pd.DataFrame(rows)
    if not df.empty:
        fig = px.box(
            df,
            x="Model",
            y="Severity",
            color="Model",
            title="Fragility Severity Distribution (N_cf_fail / N_cf)",
            points="all",
        )
        fig.show()
    return df


print("plot_severity_distribution() ready")

plot_severity_distribution() ready


## Pattern Analysis

In [6]:
from collections import Counter


def analyze_patterns(families: list[FamilyOutcome]) -> pd.DataFrame:
    pattern_counts = Counter(f.pattern for f in families)
    rows = [{"Pattern": str(p), "Count": c, "Pct (%)": round(c / len(families) * 100, 1)} for p, c in pattern_counts.most_common()]
    return pd.DataFrame(rows)


print("analyze_patterns() ready")

analyze_patterns() ready


## Counterfactual Consistency (4-way)

In [7]:
def plot_consistency(families_by_model: dict[str, list[FamilyOutcome]]) -> pd.DataFrame:
    """Four-way consistency counted over every base->CF pair."""
    order = ["both-correct", "base-correct-cf-incorrect", "both-incorrect", "base-incorrect-cf-correct"]
    rows = []
    for model, families in families_by_model.items():
        dist = consistency_distribution(families)
        total = sum(dist.values()) or 1
        for outcome in order:
            rows.append({"Model": model, "Outcome": outcome, "Count": dist[outcome], "Pct": round(100 * dist[outcome] / total, 1)})
    df = pd.DataFrame(rows)
    fig = px.bar(
        df, x="Outcome", y="Count", color="Model", barmode="group",
        title="Counterfactual Consistency (per base->CF pair)",
        category_orders={"Outcome": order},
    )
    fig.show()
    return df.pivot_table(index="Model", columns="Outcome", values="Count", fill_value=0)[order]


print("plot_consistency() ready")

plot_consistency() ready


## Correctness Drop (base -> CF)

In [8]:
def correctness_drop_table(families_by_model: dict[str, list[FamilyOutcome]]) -> pd.DataFrame:
    """Pair-level P(CF fail | base correct), shown next to family-level fragility."""
    rows = []
    for model, families in families_by_model.items():
        rows.append({
            "Model": model,
            "Correctness Drop (%)": round(100 * correctness_drop(families), 1),
            "Fragility Rate (%)": round(100 * fragility_rate(families), 1),
        })
    return pd.DataFrame(rows)


print("correctness_drop_table() ready")

correctness_drop_table() ready


## Sensitivity Score & Patch Distance

In [9]:
def pairs_to_df(pairs) -> pd.DataFrame:
    rows = []
    for p in pairs:
        pd_ = patch_distance(p)
        rows.append({
            "family_id": p.family_id,
            "cf": p.cf_instance_id,
            "base_pass": int(p.base_passed),
            "cf_pass": int(p.cf_passed),
            "spec_delta": round(p.spec_delta, 3),
            "output_delta": round(p.output_delta, 3),
            "sensitivity": round(pd_.ratio, 3) if pd_.ratio is not None else None,
            "reaction": pd_.reaction.value if pd_.reaction else None,
            "unrelated_rewrite": pd_.unrelated_rewrite,
        })
    return pd.DataFrame(rows)


def plot_sensitivity(pairs_by_model: dict) -> pd.DataFrame:
    """Mean sensitivity (S = output change / spec change) and reaction breakdown."""
    order = ["under-reaction", "appropriate", "over-reaction"]
    summary, dist_rows = [], []
    for model, pairs in pairs_by_model.items():
        ms = mean_sensitivity(pairs)
        md = median_sensitivity(pairs)
        summary.append({
            "Model": model,
            "Mean Sensitivity": round(ms, 3) if ms is not None else None,
            "Median Sensitivity": round(md, 3) if md is not None else None,
            "Pairs (spec changed)": sum(1 for p in pairs if sensitivity_score(p) is not None),
            "Unrelated rewrites": sum(1 for p in pairs if patch_distance(p).unrelated_rewrite),
        })
        dist = reaction_distribution(pairs)
        for r in order:
            dist_rows.append({"Model": model, "Reaction": r, "Count": dist[r]})
    if any(d["Count"] for d in dist_rows):
        fig = px.bar(
            pd.DataFrame(dist_rows), x="Reaction", y="Count", color="Model", barmode="group",
            title="Output reaction to spec change (S<1 under, ~1 appropriate, S>1 over)",
            category_orders={"Reaction": order},
        )
        fig.show()
    return pd.DataFrame(summary)


print("pairs_to_df() / plot_sensitivity() ready")

pairs_to_df() / plot_sensitivity() ready


## Run All Analysis

In [10]:
# Family outcome table
for model, families in families_by_model.items():
    print(f"\n=== {model} ({len(families)} families) ===")
    df = families_to_df(families)
    display(df)

# Family type distribution
type_df = plot_family_type_distribution(families_by_model)
display(type_df)

# Fragility rate
frag_df = plot_fragility_rate(families_by_model)
display(frag_df)

# Severity distribution
sev_df = plot_severity_distribution(families_by_model)

# Pattern analysis
for model, families in families_by_model.items():
    print(f"\n=== {model} patterns ===")
    pat_df = analyze_patterns(families)
    display(pat_df)

# Counterfactual consistency (4-way)
cons_df = plot_consistency(families_by_model)
display(cons_df)

# Correctness drop (base -> CF)
display(correctness_drop_table(families_by_model))

# Sensitivity score & patch distance
sens_df = plot_sensitivity(pairs_by_model)
display(sens_df)
for model, pairs in pairs_by_model.items():
    print(f"\n=== {model}: per-pair sensitivity (first 20) ===")
    display(pairs_to_df(pairs).head(20))


=== Claude Opus 4.6 (71 families) ===


,family_id,layer,base,pattern,type,fragile,severity,cf_total,cf_fail_count,cf1,cf2,cf3
0,microsoftInternal__NAV-174087,unclassified,1,"(1, 1, 1)",stable-correct,0,0.000000,2,0,1,1.0,NaN
1,microsoftInternal__NAV-174794,unclassified,1,"(1, 1, 0, 0)",fragile,1,0.666667,3,2,1,0.0,0.0
2,microsoftInternal__NAV-175765,unclassified,1,"(1, 0, 0, 0)",fragile,1,1.000000,3,3,0,0.0,0.0
3,microsoftInternal__NAV-176082,unclassified,1,"(1, 0, 0, 1)",fragile,1,0.666667,3,2,0,0.0,1.0
4,microsoftInternal__NAV-176150,unclassified,1,"(1, 0, 1)",fragile,1,0.500000,2,1,0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
66,microsoftInternal__NAV-227219,unclassified,1,"(1, 1)",stable-correct,0,0.000000,1,0,1,NaN,NaN
67,microsoftInternal__NAV-227358,unclassified,1,"(1, 1, 1)",stable-correct,0,0.000000,2,0,1,1.0,NaN
68,microsoft__BCApps-4699,unclassified,1,"(1, 0)",fragile,1,1.000000,1,1,0,NaN,NaN
69,microsoft__BCApps-4822,unclassified,1,"(1, 1, 1)",stable-correct,0,0.000000,2,0,1,1.0,NaN



=== Claude Sonnet 4.6 (71 families) ===


,family_id,layer,base,pattern,type,fragile,severity,cf_total,cf_fail_count,cf1,cf2,cf3
0,microsoftInternal__NAV-174087,unclassified,1,"(1, 1, 0)",fragile,1,0.500000,2,1,1,0.0,NaN
1,microsoftInternal__NAV-174794,unclassified,1,"(1, 1, 1, 1)",stable-correct,0,0.000000,3,0,1,1.0,1.0
2,microsoftInternal__NAV-175765,unclassified,1,"(1, 1, 0, 0)",fragile,1,0.666667,3,2,1,0.0,0.0
3,microsoftInternal__NAV-176082,unclassified,1,"(1, 0, 0, 1)",fragile,1,0.666667,3,2,0,0.0,1.0
4,microsoftInternal__NAV-176150,unclassified,1,"(1, 1, 0)",fragile,1,0.500000,2,1,1,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
66,microsoftInternal__NAV-227219,unclassified,1,"(1, 1)",stable-correct,0,0.000000,1,0,1,NaN,NaN
67,microsoftInternal__NAV-227358,unclassified,1,"(1, 1, 1)",stable-correct,0,0.000000,2,0,1,1.0,NaN
68,microsoft__BCApps-4699,unclassified,1,"(1, 0)",fragile,1,1.000000,1,1,0,NaN,NaN
69,microsoft__BCApps-4822,unclassified,1,"(1, 0, 1)",fragile,1,0.500000,2,1,0,1.0,NaN



=== GPT-5-4 (71 families) ===


...

,family_id,cf,base_pass,cf_pass,spec_delta,output_delta,sensitivity,reaction,unrelated_rewrite
0,microsoftInternal__NAV-174087,microsoftInternal__NAV-174087__cf-1,1,1,0.000,0.850,NaN,None,False
1,microsoftInternal__NAV-174087,microsoftInternal__NAV-174087__cf-2,1,1,0.012,0.903,72.258,over-reaction,False
2,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-1,1,1,0.048,0.500,10.500,over-reaction,False
3,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-2,1,0,0.024,0.500,21.000,over-reaction,True
4,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-3,1,0,0.000,0.500,NaN,None,False
5,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-1,1,0,0.052,0.905,17.552,over-reaction,True
6,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-2,1,0,0.046,0.933,20.222,over-reaction,True
7,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-3,1,0,0.051,0.957,18.748,over-reaction,True
8,microsoftInternal__NAV-176082,microsoftInternal__NAV-176082__cf-1,1,0,0.000,0.371,NaN,None,False
9,microsoftInternal__NAV-176082,microsoftInternal__NAV-176082__cf-2,1,0,0.000,0.655,NaN,None,False



=== Claude Sonnet 4.6: per-pair sensitivity (first 20) ===


,family_id,cf,base_pass,cf_pass,spec_delta,output_delta,sensitivity,reaction,unrelated_rewrite
0,microsoftInternal__NAV-174087,microsoftInternal__NAV-174087__cf-1,1,1,0.000,0.786,NaN,None,False
1,microsoftInternal__NAV-174087,microsoftInternal__NAV-174087__cf-2,1,0,0.012,0.900,72.000,over-reaction,True
2,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-1,1,1,0.048,0.000,0.000,under-reaction,False
3,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-2,1,1,0.024,0.000,0.000,under-reaction,False
4,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-3,1,1,0.000,0.500,NaN,None,False
5,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-1,1,1,0.052,0.000,0.000,under-reaction,False
6,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-2,1,0,0.046,0.933,20.222,over-reaction,True
7,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-3,1,0,0.051,0.933,18.293,over-reaction,True
8,microsoftInternal__NAV-176082,microsoftInternal__NAV-176082__cf-1,1,0,0.000,0.420,NaN,None,False
9,microsoftInternal__NAV-176082,microsoftInternal__NAV-176082__cf-2,1,0,0.000,0.516,NaN,None,False



=== GPT-5-3 Codex: per-pair sensitivity (first 20) ===


,family_id,cf,base_pass,cf_pass,spec_delta,output_delta,sensitivity,reaction,unrelated_rewrite
0,microsoftInternal__NAV-174087,microsoftInternal__NAV-174087__cf-1,1,1,0.000,0.818,NaN,None,False
1,microsoftInternal__NAV-174087,microsoftInternal__NAV-174087__cf-2,1,0,0.012,0.877,70.123,over-reaction,True
2,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-1,1,0,0.048,0.750,15.750,over-reaction,True
3,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-2,1,1,0.024,0.310,13.034,over-reaction,False
4,microsoftInternal__NAV-174794,microsoftInternal__NAV-174794__cf-3,1,1,0.000,0.310,NaN,None,False
5,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-1,0,0,0.052,0.903,17.523,over-reaction,True
6,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-2,0,0,0.046,0.857,18.571,over-reaction,True
7,microsoftInternal__NAV-175765,microsoftInternal__NAV-175765__cf-3,0,0,0.051,0.957,18.748,over-reaction,True
8,microsoftInternal__NAV-176082,microsoftInternal__NAV-176082__cf-1,1,0,0.000,0.651,NaN,None,False
9,microsoftInternal__NAV-176082,microsoftInternal__NAV-176082__cf-2,1,0,0.000,0.710,NaN,None,False
